# Visual Prompt Refinement with Grok

Image generation workflows often start with a rough idea: a reference photo, a style direction, or a short prompt. The hard part is turning that loose input into a precise visual brief that a downstream image model can use consistently.

In this cookbook, we use Grok's image understanding and structured outputs to inspect reference images, extract visual attributes, and build generator-ready prompts. The final prompt can be used with image generation systems such as Grok image generation, FLUX, Stable Diffusion, or an internal creative pipeline.

The workflow is useful for creative teams that need repeatable prompt writing, catalog enrichment, image QA, or style-consistent production at scale.

## Table of Contents

- [Setup](#setup)
- [Load a Reference Image](#load-a-reference-image)
- [Describe the Image](#describe-the-image)
- [Extract a Structured Visual Brief](#extract-a-structured-visual-brief)
- [Turn the Brief into a Generation Prompt](#turn-the-brief-into-a-generation-prompt)
- [Batch Processing](#batch-processing)
- [Conclusion](#conclusion)

## Setup

Install the small set of dependencies used in this notebook. The `openai` SDK is used with xAI's OpenAI-compatible API.

In [ ]:
%pip install --quiet openai python-dotenv pydantic pandas tqdm IPython

> **Note:** Set `XAI_API_KEY` in your environment or in a `.env` file at the root of this repo before running API calls.

In [ ]:
import base64
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()

BASE_URL = "https://api.x.ai/v1"
GROK_VISION_MODEL = "grok-2-vision-latest"
GROK_TEXT_MODEL = "grok-4"

client = OpenAI(
    base_url=BASE_URL,
    api_key=os.getenv("XAI_API_KEY"),
)

In [ ]:
def base64_encode_image(image_path: str | Path) -> str:
    image_path = Path(image_path)
    with image_path.open("rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def image_message(prompt: str, image_path: str | Path) -> list[dict]:
    encoded_image = base64_encode_image(image_path)
    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{encoded_image}",
                    },
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]

## Load a Reference Image

For a portable example, this notebook reuses an image already included in the structured data extraction cookbook. You can replace `REFERENCE_IMAGE` with any local image path.

In [ ]:
from IPython.display import Image, display

REFERENCE_IMAGE = Path("../multimodal/structured_data_extraction/data/images/000001.jpg")
display(Image(filename=str(REFERENCE_IMAGE)))

## Describe the Image

Start with an open-ended description. This is helpful when you want to see what the model notices before imposing a schema.

In [ ]:
description_prompt = """
Describe this image for a creative production team.
Focus on visible subject matter, composition, lighting, color palette, camera feel, styling, background, and any quality issues.
Do not identify the person. Do not infer sensitive attributes.
"""

response = client.chat.completions.create(
    model=GROK_VISION_MODEL,
    messages=image_message(description_prompt, REFERENCE_IMAGE),
)

print(response.choices[0].message.content)

## Extract a Structured Visual Brief

A structured brief makes the output easier to audit, store, compare, and feed into another system. It also gives you clean metadata for batch pipelines.

In [ ]:
class VisualBrief(BaseModel):
    subject: str = Field(description="Short neutral description of the visible subject matter")
    setting: str = Field(description="Physical environment or background")
    composition: str = Field(description="Framing, perspective, crop, and subject placement")
    lighting: str = Field(description="Lighting direction, softness, contrast, and mood")
    color_palette: list[str] = Field(description="Dominant colors and tones")
    styling: list[str] = Field(description="Visible wardrobe, props, materials, or design cues")
    camera_feel: str = Field(description="Lens, depth of field, grain, sharpness, or photographic feel")
    quality_notes: list[str] = Field(description="Artifacts, blur, compression, text overlays, or retouching needs")
    reusable_tags: list[str] = Field(description="Searchable tags useful for retrieval or generation")


structured_prompt = f"""
Analyze the image and return a structured visual brief that follows this JSON schema:
{json.dumps(VisualBrief.model_json_schema(), indent=2)}

Rules:
- Describe only visible content.
- Do not identify the person.
- Do not infer sensitive attributes.
- Keep the wording useful for creative production and prompt engineering.
"""

In [ ]:
brief_completion = client.beta.chat.completions.parse(
    model=GROK_VISION_MODEL,
    messages=image_message(structured_prompt, REFERENCE_IMAGE),
    response_format=VisualBrief,
)

visual_brief = brief_completion.choices[0].message.parsed
print(visual_brief.model_dump_json(indent=2))

## Turn the Brief into a Generation Prompt

Now pass the structured brief to a text model and ask it to create a generator-ready prompt. This separates observation from prompt writing: the vision model extracts facts, and the text model turns those facts into a clean creative instruction.

In [ ]:
class GenerationPrompt(BaseModel):
    positive_prompt: str = Field(description="A polished image generation prompt")
    negative_prompt: str = Field(description="Things the generator should avoid")
    style_notes: list[str] = Field(description="Reusable style controls")
    suggested_aspect_ratio: str = Field(description="Recommended aspect ratio, such as 1:1, 4:5, or 16:9")
    suggested_parameters: dict[str, str | int | float] = Field(description="Optional generation parameters")


prompt_writer_instruction = f"""
You are a senior image prompt engineer.
Create a generator-ready prompt from this visual brief:
{visual_brief.model_dump_json(indent=2)}

Return JSON that follows this schema:
{json.dumps(GenerationPrompt.model_json_schema(), indent=2)}

Requirements:
- Preserve the core composition, lighting, and visual style.
- Write a prompt that is specific but not overloaded.
- Avoid naming real people, brands, or copyrighted characters.
- Use neutral, production-safe wording.
"""

prompt_completion = client.beta.chat.completions.parse(
    model=GROK_TEXT_MODEL,
    messages=[{"role": "user", "content": prompt_writer_instruction}],
    response_format=GenerationPrompt,
)

generation_prompt = prompt_completion.choices[0].message.parsed
print(generation_prompt.model_dump_json(indent=2))

## Batch Processing

The same pattern can be used for folders of images. Below is a compact batch function that produces one structured brief per image. In a production pipeline, you would add retries, rate limits, and persistent storage.

In [ ]:
from tqdm import tqdm


def extract_visual_brief(image_path: str | Path) -> VisualBrief:
    completion = client.beta.chat.completions.parse(
        model=GROK_VISION_MODEL,
        messages=image_message(structured_prompt, image_path),
        response_format=VisualBrief,
    )
    return completion.choices[0].message.parsed


sample_images = sorted(Path("../multimodal/structured_data_extraction/data/images").glob("*.jpg"))[:5]
batch_results = []

for image_path in tqdm(sample_images):
    brief = extract_visual_brief(image_path)
    batch_results.append({"image": image_path.name, **brief.model_dump()})

batch_results[:2]

In [ ]:
import pandas as pd

df = pd.DataFrame(batch_results)
df.to_json("visual_briefs.jsonl", orient="records", lines=True)
df.head()

## Conclusion

This workflow turns unstructured reference images into reusable creative production assets:

1. Grok describes the image.
2. Grok extracts a structured visual brief.
3. Grok converts the brief into a generator-ready prompt.
4. The same approach scales to folders of images for metadata, prompt libraries, QA, or creative automation.

The key idea is to keep each step explicit: observe first, structure second, generate third. That makes the pipeline easier to debug, evaluate, and improve.